In [ ]:
#Load Silver + Policy + Claims + Narrative + Driver
import os
import pandas as pd
from decimal import Decimal
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

def make_engine(user, pw):
    host    = os.getenv("ORACLE_BRONZE_HOST",    "localhost")
    port    = os.getenv("ORACLE_BRONZE_PORT",    "1521")
    service = os.getenv("ORACLE_BRONZE_SERVICE", "FREEPDB1")
    return create_engine(f"oracle+oracledb://{user}:{pw}@{host}:{port}/?service_name={service}")

engine_bronze = make_engine(
    os.getenv("ORACLE_BRONZE_USER",    "claims_bronze"),
    os.getenv("ORACLE_BRONZE_PASSWORD","claims_bronze")
)
engine_silver = make_engine(
    os.getenv("ORACLE_SILVER_USER",    "claims_silver"),
    os.getenv("ORACLE_SILVER_PASSWORD","claims_silver")
)
engine_gold = make_engine(
    os.getenv("ORACLE_GOLD_USER",      "claims_gold"),
    os.getenv("ORACLE_GOLD_PASSWORD",  "claims_gold")
)

with engine_silver.connect() as conn:
    df_silver = pd.read_sql("SELECT * FROM claim_evidence_summary", conn)

with engine_gold.connect() as conn:
    df_policy = pd.read_sql("SELECT * FROM policies", conn)
    df_claims = pd.read_sql("SELECT claim_id FROM claims", conn)
    df_driver = pd.read_sql("SELECT * FROM drivers", conn)

with engine_bronze.connect() as conn:
    df_inbox = pd.read_sql(
        "SELECT claim_id_ext, policy_id, narrative FROM inbound_claims", conn
    )

# Aggregate findings per claim
df_findings = (
    df_silver.groupby("claim_id")["findings"]
    .apply(lambda x: "\n".join(x))
    .reset_index()
    .rename(columns={"findings": "aggregated_findings"})
)

# Join Silver → inbound_claims (get policy_id + narrative) → Policy → Driver → Claims
df_joined = (
    df_silver
    .merge(df_inbox, left_on="claim_id", right_on="claim_id_ext", how="left")
    .merge(
        df_policy[["policy_id", "status", "limit_property_usd",
                   "driver_id", "vehicle_make", "vehicle_model", "vehicle_year"]],
        on="policy_id", how="left"
    )
    .merge(df_driver[["driver_id", "full_name", "license_number"]], on="driver_id", how="left")
    .merge(
        df_claims.rename(columns={"claim_id": "existing_claim_id"}),
        left_on="claim_id", right_on="existing_claim_id", how="left"
    )
    .merge(df_findings, on="claim_id", how="left")
)

df_pd = df_joined.copy()
print(f"Loaded {len(df_pd)} evidence records for {df_pd['claim_id'].nunique()} claims")


In [ ]:
#Define Rules Prompt
RULES = """
You are an insurance claims decisioning assistant. Apply the following rules to determine the claim decision:

Rule ID | Condition | Decision
R1 | Policy is not active in gold.policy | REJECT
R2 | Claim already exists in gold.claims | MANUAL_REVIEW
R3 | Evidence confidence < 0.70 | MANUAL_REVIEW
R4 | Modality is video and confidence ≥ 0.85 | APPROVE_FAST_TRACK
R5 | Modality is image and confidence ≥ 0.80 | APPROVE
R6 | Default fallback | MANUAL_REVIEW

Return a JSON with keys:
- decision: one of [REJECT, MANUAL_REVIEW, APPROVE, APPROVE_FAST_TRACK]
- fusion_text: detailed summary of why this decision was made
- reasons_json: one-line reason summary
- evidence_refs_json: list of evidence URIs
- est_payout_usd: any amount less than limit_property_usd
- action: one of [NOTIFY, ESCALATE, PAYOUT, ARCHIVE]
"""


In [ ]:
#Initialize AI Client — Ollama (on-premises)
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

ollama_client = OpenAI(
    base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
    api_key="ollama"  # required by SDK, not used by Ollama
)
OLLAMA_TEXT_MODEL = os.getenv("OLLAMA_TEXT_MODEL", "qwen2.5:7b")
print(f"Ollama client initialized → {os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1')}")
print(f"Text model: {OLLAMA_TEXT_MODEL}")


In [ ]:
#Claim-Level Decision Function with Fusion
import json
from datetime import datetime
from collections import defaultdict

grouped = defaultdict(list)
for _, row in df_pd.iterrows():
    grouped[row.claim_id].append(row)

def run_claim_level_llm(claim_id, rows):
    try:
        modalities         = [r.modality for r in rows]
        confidences        = [float(r.confidence) for r in rows]
        uris               = [r.source_uri for r in rows]
        policy_status      = rows[0].status
        existing_claim     = "Yes" if rows[0].existing_claim_id else "No"
        limit_property_usd = rows[0].limit_property_usd
        narrative          = rows[0].narrative or "No narrative provided."
        aggregated_findings = rows[0].aggregated_findings or "No findings available."
        driver_name        = rows[0].full_name or "Unknown"
        license_number     = rows[0].license_number or "Unknown"
        vehicle_make       = rows[0].vehicle_make or "Unknown"
        vehicle_model      = rows[0].vehicle_model or "Unknown"
        vehicle_year       = rows[0].vehicle_year or "Unknown"

        prompt = f"""
{RULES}

This summary will be read by a business user, not a technical analyst. Please ensure the response is clear, concise, and business-centric — focusing on what happened in the claim, why the decision was made, and what actions are recommended. Avoid technical jargon and use confidence score along with other RULES for decision but not populate the value in fusion_text and emphasize clarity and relevance.

Respond ONLY with a valid JSON object. No markdown, no explanation outside JSON.

Claim ID: {claim_id}
Policy Active: {policy_status}
Existing Claim: {existing_claim}
Modalities: {modalities}
Confidences: {confidences}
Evidence URIs: {uris}
Limit Property USD: {limit_property_usd}

Narrative:
{narrative}

Driver Details:
Name: {driver_name}
License: {license_number}

Vehicle Details:
Make: {vehicle_make}
Model: {vehicle_model}
Year: {vehicle_year}

Evidence Summary:
{aggregated_findings}
"""

        resp = ollama_client.chat.completions.create(
            model=OLLAMA_TEXT_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=4000,
            temperature=0.3,
            response_format={"type": "json_object"}
        )
        content = resp.choices[0].message.content
        parsed = json.loads(content)

        return (
            claim_id,
            parsed.get("decision", "MANUAL_REVIEW"),
            parsed.get("fusion_text", ""),
            parsed.get("action", "NOTIFY"),
            parsed.get("reasons_json", ""),
            json.dumps(uris),
            Decimal(str(max(confidences))),
            Decimal(str(parsed.get("est_payout_usd", "0.0"))),
            datetime.utcnow(),
            "system",
            datetime.utcnow(),
            "system"
        )

    except Exception as e:
        return (
            claim_id,
            "MANUAL_REVIEW",
            f"LLM failed: {str(e)}",
            "NOTIFY",
            "LLM error",
            json.dumps(uris),
            Decimal(str(max(confidences))),
            Decimal("0.0"),
            datetime.utcnow(),
            "system",
            datetime.utcnow(),
            "system"
        )


In [ ]:
#Run Inference in Parallel
from concurrent.futures import ThreadPoolExecutor, as_completed

records = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(run_claim_level_llm, claim_id, rows) for claim_id, rows in grouped.items()]
    for f in as_completed(futures):
        records.append(f.result())


In [ ]:
#Normalize Results to DataFrame
import pandas as pd

df_pd_out = pd.DataFrame(records, columns=[
    "claim_id", "decision", "fusion_text", "action", "reasons_json",
    "evidence_refs_json", "confidence", "est_payout_usd",
    "created_at", "created_by", "updated_at", "updated_by"
])

print(f"Processed {len(df_pd_out)} claims")
display(df_pd_out[["claim_id", "decision", "action", "confidence", "est_payout_usd"]])


In [ ]:
#Write to Gold Table — claim_decision (MERGE upsert)
from sqlalchemy import text

merge_sql = text("""
    MERGE INTO claim_decision t
    USING (SELECT :claim_id AS claim_id FROM dual) s
    ON (t."claim_id" = s.claim_id)
    WHEN MATCHED THEN UPDATE SET
        t."decision"           = :decision,
        t."fusion_text"        = :fusion_text,
        t."action"             = :action,
        t."reasons_json"       = :reasons_json,
        t."evidence_refs_json" = :evidence_refs_json,
        t."confidence"         = :confidence,
        t."est_payout_usd"     = :est_payout_usd,
        t."updated_at"         = :updated_at,
        t."updated_by"         = :updated_by
    WHEN NOT MATCHED THEN INSERT (
        "claim_id", "decision", "fusion_text", "action",
        "reasons_json", "evidence_refs_json", "confidence", "est_payout_usd",
        "created_at", "created_by", "updated_at", "updated_by"
    ) VALUES (
        :claim_id, :decision, :fusion_text, :action,
        :reasons_json, :evidence_refs_json, :confidence, :est_payout_usd,
        :created_at, :created_by, :updated_at, :updated_by
    )
""")

with engine_gold.connect() as conn:
    for _, row in df_pd_out.iterrows():
        conn.execute(merge_sql, {
            "claim_id":           row["claim_id"],
            "decision":           row["decision"],
            "fusion_text":        row["fusion_text"],
            "action":             row["action"],
            "reasons_json":       row["reasons_json"],
            "evidence_refs_json": row["evidence_refs_json"],
            "confidence":         float(row["confidence"]) if row["confidence"] is not None else None,
            "est_payout_usd":     float(row["est_payout_usd"]) if row["est_payout_usd"] is not None else None,
            "created_at":         row["created_at"],
            "created_by":         row["created_by"],
            "updated_at":         row["updated_at"],
            "updated_by":         row["updated_by"],
        })
    conn.commit()
print(f"Upserted {len(df_pd_out)} record(s) to claim_decision (Gold)")


In [ ]:
# View Gold — claim_decision
with engine_gold.connect() as conn:
    display(pd.read_sql("SELECT * FROM claim_decision", conn))


In [ ]:
from IPython.display import display, HTML
import json

def display_claim(df):
    row = df.iloc[0]
    reasons  = row['reasons_json']  or "-"
    evidence = row['evidence_refs_json'] or "-"
    try:
        reasons_list = json.loads(reasons) if reasons != "-" else []
    except Exception:
        reasons_list = []
    try:
        evidence_list = json.loads(evidence) if evidence != "-" else []
    except Exception:
        evidence_list = []

    reasons_html  = "<ul>" + "".join(f"<li>{r}</li>" for r in reasons_list)  + "</ul>" if reasons_list  else reasons
    evidence_html = "<ul>" + "".join(f"<li>{e}</li>" for e in evidence_list) + "</ul>" if evidence_list else "-"

    html_content = f"""
    <style>
        .claim-container {{
            font-family: 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
            max-width: 820px;
            background-color: #ffffff;
            border: 1px solid #DDE0E3;
            border-radius: 6px;
            padding: 8px 12px;
            box-shadow: 0 1px 2px rgba(0,0,0,0.05);
            margin: 8px auto;
            color: #212529;
            line-height: 1.3;
        }}
        .header {{
            display: flex;
            justify-content: space-between;
            flex-wrap: wrap;
            border-bottom: 1px solid #E9ECEF;
            padding-bottom: 2px;
            margin-bottom: 2px;
        }}
        .header-item {{
            font-size: 12px;
            font-weight: 600;
            color: #495057;
            margin-bottom: 0;
        }}
        .header-item strong {{ font-weight: 700; color: #212529; }}
        .section {{ display: flex; gap: 6px; margin-bottom: 4px; }}
        .section-title {{
            flex: 0 0 70px;
            font-weight: 600;
            font-size: 12px;
            color: #343A40;
            margin-top: 4px;
        }}
        .content-box, .evidence-box {{
            background: #F8F9FA;
            border: 1px solid #E9ECEF;
            border-radius: 3px;
            padding: 6px 8px;
            font-size: 11px;
            color: #495057;
            white-space: pre-wrap;
            flex: 1;
            max-height: 120px;
            overflow-y: auto;
        }}
        ul {{ padding-left: 18px; margin: 0; }}
        li {{ margin-bottom: 3px; }}
        .metadata-container {{
            display: flex;
            flex-wrap: wrap;
            gap: 6px;
            font-size: 10px;
            color: #6C757D;
            border-top: 1px solid #E9ECEF;
            padding-top: 2px;
            margin-top: 2px;
        }}
        .metadata-item {{ min-width: 140px; font-weight: 500; }}
    </style>

    <div class="claim-container">
        <div class="header">
            <div class="header-item">Claim ID: <strong>{row['claim_id']}</strong></div>
            <div class="header-item">Decision: <strong style="color:#198754;">{row['decision']}</strong></div>
            <div class="header-item">Action: <strong style="color:#0D6EFD;">{row['action']}</strong></div>
            <div class="header-item">Payout: <strong style="color:#D63384;">${float(row['est_payout_usd'] or 0):.2f}</strong></div>
        </div>
        <div class="section">
            <span class="section-title">Summary:</span>
            <div class="content-box">{row['fusion_text']}</div>
        </div>
        <div class="section">
            <span class="section-title">Reasons:</span>
            <div class="content-box">{reasons_html}</div>
        </div>
        <div class="section">
            <span class="section-title">Evidence:</span>
            <div class="evidence-box">{evidence_html}</div>
        </div>
        <div class="metadata-container">
            <div class="metadata-item">Confidence: {float(row['confidence'] or 0) * 100:.2f}%</div>
            <div class="metadata-item">Created: {row['created_at']} by {row['created_by']}</div>
            <div class="metadata-item">Updated: {row['updated_at']} by {row['updated_by']}</div>
        </div>
    </div>
    """
    display(HTML(html_content))


In [ ]:
# Export to CSV
output_path = "claim_decision_export.csv"
df_pd_out.to_csv(output_path, index=False)
print(f"Exported to {output_path}")


In [ ]:
# Display latest claim decision
with engine_gold.connect() as conn:
    df_gold_view = pd.read_sql(
        "SELECT * FROM claim_decision WHERE ROWNUM <= 1", conn
    )

if not df_gold_view.empty:
    display_claim(df_gold_view)


In [ ]:
print("****Enriched and  data loaded to Gold layer Successfully ****")